# Single-basin parameter sweep
Before committing to LHS / Optuna / JAX, this notebook tests two things on ONE basin:
1. Linearity: do the basin medians move (nearly) linearly with each of the 4 knobs?
2. Influence: how far does each knob move the medians across its range? (`Fwc` may barely move the basin scale.)

We report influence (marginal, one-at-a-time sensitivity), NOT identifiability. A knob can be
influential yet non-identifiable if its effect is collinear with another knob's; that shows up only
in the joint (2-D) sweep in Section 5, never in a 1-D slice.

> Data.
> `.rds` on NAS). Until then this runs on the synthetic toy basin. The GLORICH targets ARE in the
> repo (`data/HUC4_calibration.csv`). Absolute pCO2 values here are illustrative; read the PATTERNS.

## 0. Setup

In [ ]:
import os, sys
_found = False
for p in ['.', '..']:
    if os.path.isdir(os.path.join(p, 'conus_carbon')):
        sys.path.insert(0, os.path.abspath(p)); _found = True; break
if not _found:
    raise RuntimeError('Run this notebook from the CONUS_carbon-master_python/ directory '
                       '(or its notebooks/ subfolder) so the conus_carbon package is importable.')
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from conus_carbon import synthetic_basin, sweep, io, calibrate
from conus_carbon.constants import PARAM_NAMES, C_GROUNDWATER_PPM
print('knobs:', PARAM_NAMES)

## 1. Load the basin + targets

Right now we use the synthetic toy basin (`huc4=None`). When a real prepared basin is available, change the three marked lines together: `huc4`, `basin`, and `targets`. Everything below
is unchanged. Note: this sweep treats the basin in isolation (no cross-basin upstream imports);
that is correct for a headwater test basin, but a downstream basin would need `upstream_df`.


In [ ]:
# --- SYNTHETIC (now)
# Change these THREE lines together for a real basin. huc4 is a 4-digit string ('0107').
# If its HUC2 is 04 or 16, Fwc_lake auto-switches to photosynthesis bounds; the five Great-Lakes
# codes (0418/0419/0424/0426/0428) are excluded (the sweep raises), matching the R pipeline.
huc4 = None                                        # (1) real HUC4, e.g. '0107'
basin = synthetic_basin.make_synthetic_basin()     # (2) real: io.load_basin_hydrography('basin_0107.parquet')
targets = synthetic_basin.SYNTHETIC_TARGETS        # (3) real: io.load_calibration_targets(huc4)

print('basin reaches:', len(basin), '| targets:', targets)
print('river reaches:', int((basin.waterbody=='River').sum()),
      '| lake reaches:', int((basin.waterbody=='Lake/Reservoir').sum()))

## 2. One-at-a-time (OAT) sweep of all four knobs

Vary each knob across its bounds (others held at the midpoint baseline) and record the
medians. The summary reports, per knob: `influence_ppm` (how much its OWN water-body median moves),
`cross_influence_ppm` (how much it moves the OTHER water-body median, i.e. river/lake coupling),
`linearity_R2` (NaN if the knob is flat), and a `flat` flag.

> These spans are at an uncalibrated midpoint baseline, so an absolute median can be
> nonphysical (e.g. a lake median far above the 16,000 ppm groundwater source). Read the shape and
> the influence, not the absolute ppm.

In [ ]:
baseline = sweep.default_baseline(huc4)   # midpoint; UNCALIBRATED operating point (see caveats)
sweeps = sweep.sweep_all_1d(basin, targets['riverCO2'], targets['lakeCO2'],
                            baseline=baseline, n=25, huc4=huc4)
summary = sweep.summarize_1d(sweeps)
print(summary.to_string(index=False))

## 3. Plot: is it linear? how much does each knob move things?

Each panel plots the knob's own water-body median vs the knob. Straight line = linear along
this axis; large vertical span = high influence. A knob whose line is nearly flat has low influence,
so its median cannot pin it down.


In [ ]:
fig, axes = plt.subplots(2,2,figsize=(10,6.5)); axes=axes.ravel()
for ax, name in zip(axes, PARAM_NAMES):
    df = sweeps[name]; rc = sweep.OWN_RESPONSE[name]
    ax.plot(df['value'], df[rc], 'o-', color='#2563eb', ms=3)
    infl = sweep.influence_ppm(df, rc); r2 = sweep.linearity_r2(df, rc)
    r2s = 'flat' if not np.isfinite(r2) else f'R2={r2:.4f}'
    ax.set_title(f'{name}  (moves {infl:,.0f} ppm, {r2s})', fontsize=10)
    ax.set_xlabel(name); ax.set_ylabel(rc + ' [ppm]'); ax.grid(alpha=0.25)
fig.suptitle('OAT sweep: own-water-body median vs each knob', fontweight='bold')
plt.tight_layout(); plt.show()

## 4. Read-out

EN.
- Influence, not identifiability. `influence_ppm` is a marginal, one-at-a-time sensitivity. A
  low value means the median cannot pin the knob down; a high value does NOT prove identifiability,
  because two knobs can be collinear (Section 5 checks that).
- Matt's `Fwc` suspicion is river-specific. Expect `Fwc_riv` to move the river median very
  little (short river residence time) while `Fwc_lake` can move the lake median a lot (long lake
  residence). So the point is the river/lake split, not `Fwc` as a whole.
- Marginal linearity is necessary, not sufficient. Even if all four 1-D slices look linear, the
  joint surface can still curve through interactions; and with 4 knobs vs 2 targets the fit is
  underdetermined, so equifinality persists regardless of linearity. Confirm joint behavior in §5.


In [ ]:
tbl = summary.dropna(subset=['influence_ppm']).copy()
if len(tbl):
    least = tbl.loc[tbl['influence_ppm'].idxmin()]
    most  = tbl.loc[tbl['influence_ppm'].idxmax()]
    print(f"lowest-influence knob : {least['knob']} ({least['influence_ppm']:,.1f} ppm over its range)")
    print(f"highest-influence knob: {most['knob']} ({most['influence_ppm']:,.1f} ppm over its range)")
print('flat (near-zero influence) knobs:', list(summary.loc[summary['flat'],'knob']) or 'none')
print('note: influence spans are over each knob\'s own bounds, which differ in magnitude',
      '(Cbz 0..2000 ppm vs Fwc 0..1e-2 ppm/s), so a small input range does NOT mean unimportant;',
      'judge each knob by its ppm span. influence_ppm (ppm of response) IS comparable across knobs.')

## 5. Equifinality: a same-body 2-D sweep

OAT slices cannot show joint trade-offs. Equifinality is a trade-off between two knobs that
drive the same target: `Cbz_lake` (streambed) and `Fwc_lake` (water-column) BOTH add CO2 to lake
reaches, so they can trade off (more of one, less of the other, same lake median) toward a ridge.

We threshold the behavioral set on the full 2-target cost = |river err| + |lake err|, the same
objective the calibration uses. Do NOT threshold on the lake term alone: lake knobs also move the
river median, because river reaches downstream of a lake inherit its CO2 through routing (this is
the `cross_influence_ppm` from Section 2). We print the river-median span below to make that coupling
explicit.

> On the synthetic basin the GLORICH targets are not reachable on this slice, so the optimum
> pins to a corner and you see a gradient, not an interior ridge. Read this section as the **method
> scaffold** (how the behavioral set is computed), NOT as an identifiability result. On a real basin
> with reachable targets, a diagonal low-cost valley = equifinal; a single tight minimum = identifiable.

In [ ]:
# choose (i,j) to match the water-body under study; (1,3)=lake pair needs lake reaches present.
# river-only basin -> use a river pair like (0,2).
i, j = 1, 3   # Cbz_lake (1) x Fwc_lake (3): two lake sources that can trade off
surf = sweep.sweep_2d(basin, targets['riverCO2'], targets['lakeCO2'], baseline, i, j, n=25, huc4=huc4)
beh = sweep.behavioral_set(surf, abs_tol_ppm=50.0)   # threshold on the FULL 2-target 'cost'
xi, xj = PARAM_NAMES[i], PARAM_NAMES[j]
piv = surf.pivot(index=xj, columns=xi, values='cost')
fig, ax = plt.subplots(figsize=(7,5.2))
im = ax.pcolormesh(piv.columns, piv.index, piv.values, shading='auto', cmap='viridis_r')
ax.scatter(beh[xi], beh[xj], s=14, color='#e11d48', label='behavioral (cost <= best + 50 ppm)')
fig.colorbar(im, ax=ax, label='full 2-target cost [ppm]'); ax.set_xlabel(xi); ax.set_ylabel(xj)
ax.set_title(f'2-D cost surface: {xi} x {xj}'); ax.legend(loc='upper right', fontsize=8)
plt.tight_layout(); plt.show()
river_span = sweep.influence_ppm(surf, 'river_median')
lake_lo, lake_hi = surf['lake_median'].min(), surf['lake_median'].max()
sep = ('so lake and river params are NOT separable here' if river_span >= sweep.INFLUENCE_FLOOR_PPM
       else 'so lake knobs barely reach the river median here (nearly separable on this slice)')
print(f"slice best cost = {beh.attrs['slice_best_cost']:.1f} ppm | behavioral points: {len(beh)}/{len(surf)}")
print(f"coupling: lake knobs move the RIVER median by {river_span:,.0f} ppm on this slice ({sep}).")
print(f"reachable lake_median on this slice: {lake_lo:,.0f}..{lake_hi:,.0f} ppm; target = {targets['lakeCO2']:.0f}",
      "-> on the synthetic basin the target is out of range, so this is a method scaffold, not an",
      "identifiability result.")

## 6. Caveats + next steps

EN.
- Synthetic numbers. Absolute pCO2 here is illustrative. The midpoint baseline is uncalibrated,
  so a knob's slice can reach nonphysical values (e.g. a lake median far above the 16,000 ppm
  groundwater source). Read the SHAPE and the RELATIVE influence, not the absolute ppm.
- Influence != identifiability. A high `influence_ppm` does not prove a knob is identifiable;
  the 2-D slice is where collinearity (and true equifinality) shows up.
- All numbers are at ONE baseline. With interactions, a knob that looks flat at the midpoint can
  matter near a range edge. Re-check at the fitted optimum once a real basin gives one.
- Next: when the lab sends ~5 real basins, change the three lines in Section 1 and re-run. If
  `linearity_R2` stays near 1 for the influential knobs AND `Fwc_riv` stays low-influence, the fit is
  close to a small analytic problem, so heavy methods may be unnecessary just to fit one basin. This
  is about optimizer sophistication only: the 4-knob vs 2-target fit stays underdetermined (equifinal)
  regardless of linearity, and a Bayesian step still earns its place through cross-basin pooling (per
  the earlier note), not for the fit alone.
